<p align="right"><a href="./C2_W2_SoftMax.ipynb">English</a> | <strong>中文</strong></p>

# Optional Lab - Softmax 函数（Softmax Function）
在本实验中，我们将探索 softmax 函数。它在 Softmax 回归和解决多分类问题的神经网络中都会用到。

<center>  <img  src="./images/C2_W2_Softmax_Header.PNG" width="600" />  <center/>

In [3]:
import numpy as np
import matplotlib.pyplot as plt
plt.style.use('./deeplearning.mplstyle')
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from IPython.display import display, Markdown, Latex
from sklearn.datasets import make_blobs
%matplotlib widget
from matplotlib.widgets import Slider
from lab_utils_common import dlc
from lab_utils_softmax import plt_softmax
import logging
logging.getLogger("tensorflow").setLevel(logging.ERROR)
tf.autograph.set_verbosity(0)

> **注意**：本课程的 notebook 通常采用从 0 开始、到 N-1 结束的计数约定（$\sum_{i=0}^{N-1}$），而课程讲义从 1 开始、到 N 结束（$\sum_{i=1}^{N}$）。这是因为代码通常从 0 开始迭代，而讲义里从 1 数到 N 会让方程更简洁。本 notebook 的方程比一般实验多，因此打破这个约定，采用从 1 数到 N。

## Softmax 函数
无论在 softmax 回归、还是带 Softmax 输出的神经网络中，都会生成 N 个输出，并从中选出一个作为预测类别。两种情况下，都由一个线性函数生成向量 $\mathbf{z}$，再对它施加 softmax 函数。softmax 函数把 $\mathbf{z}$ 转换成一个概率分布（如下所述）。施加 softmax 后，每个输出都在 0 到 1 之间、且所有输出之和为 1，因此可解释为概率。输入越大，对应的输出概率越大。
<center>  <img  src="./images/C2_W2_SoftmaxReg_NN.png" width="600" />  

softmax 函数可写作：
$$a_j = \frac{e^{z_j}}{ \sum_{k=1}^{N}{e^{z_k} }} \tag{1}$$
输出 $\mathbf{a}$ 是一个长度为 N 的向量，所以对 softmax 回归，也可以写成：
\begin{align}
\mathbf{a}(x) =
\begin{bmatrix}
P(y = 1 | \mathbf{x}; \mathbf{w},b) \\
\vdots \\
P(y = N | \mathbf{x}; \mathbf{w},b)
\end{bmatrix}
=
\frac{1}{ \sum_{k=1}^{N}{e^{z_k} }}
\begin{bmatrix}
e^{z_1} \\
\vdots \\
e^{z_{N}} \\
\end{bmatrix} \tag{2}
\end{align}

这表明输出是一个概率向量。第一项是：在给定输入 $\mathbf{x}$ 和参数 $\mathbf{w}$、$\mathbf{b}$ 下，输入属于第一个类别的概率。
我们来写一个 NumPy 实现：

In [4]:
def my_softmax(z):
    ez = np.exp(z)              #element-wise exponenial
    sm = ez/np.sum(ez)
    return(sm)

下面用滑块改变 `z` 输入的值。

In [5]:
plt.close("all")
plt_softmax(my_softmax)

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

在你改变上面各个 z 的值时，有几点值得注意：
* softmax 分子里的指数会放大数值间的微小差异
* 各输出值之和为一
* softmax 横跨所有输出。例如改变 `z0` 会改变 `a0`-`a3` 的值。这与 ReLU 或 Sigmoid 等激活不同——那些是单输入单输出。

## 代价（Cost）
<center> <img  src="./images/C2_W2_SoftMaxCost.png" width="400" />    <center/>

与 Softmax 相关的损失函数是交叉熵损失（cross-entropy loss）：
\begin{equation}
  L(\mathbf{a},y)=\begin{cases}
    -log(a_1), & \text{if $y=1$}.\\
        &\vdots\\
     -log(a_N), & \text{if $y=N$}
  \end{cases} \tag{3}
\end{equation}

其中 y 是该样本的目标类别，$\mathbf{a}$ 是 softmax 函数的输出。特别地，$\mathbf{a}$ 中的值是和为一的概率。
>**回忆：** 本课程中，Loss 针对单个样本，Cost 覆盖所有样本。

注意上面 (3) 中，只有对应目标的那一行对损失有贡献，其他行都是零。要写出代价方程，我们需要一个 '指示函数（indicator function）'——当下标与目标匹配时为 1，否则为零。
    $$\mathbf{1}\{y == n\} = =\begin{cases}
    1, & \text{if $y==n$}.\\
    0, & \text{otherwise}.
  \end{cases}$$
于是代价为：
\begin{align}
J(\mathbf{w},b) = -\frac{1}{m} \left[ \sum_{i=1}^{m} \sum_{j=1}^{N}  1\left\{y^{(i)} == j\right\} \log \frac{e^{z^{(i)}_j}}{\sum_{k=1}^N e^{z^{(i)}_k} }\right] \tag{4}
\end{align}

其中 $m$ 是样本数，$N$ 是输出数。这是所有损失的平均。

## Tensorflow
本实验会讨论在 Tensorflow 中实现 softmax + 交叉熵损失的两种方式：'显而易见（obvious）' 的方法和 '推荐（preferred）' 的方法。前者最直接，后者数值上更稳定。

我们先创建一个数据集，用来训练一个多分类模型。

In [6]:
# make  dataset for example
centers = [[-5, 2], [-2, -2], [1, 2], [5, -2]]
X_train, y_train = make_blobs(n_samples=2000, centers=centers, cluster_std=1.0,random_state=30)

### *显而易见* 的组织方式

下面的模型把 softmax 作为最后一个 Dense 层的激活来实现，损失函数在 `compile` 指令中单独指定。

损失函数是 `SparseCategoricalCrossentropy`，即上面 (3) 描述的损失。这个模型里，softmax 发生在最后一层；损失函数接收 softmax 的输出（一个概率向量）。

In [7]:
model = Sequential(
    [ 
        Dense(25, activation = 'relu'),
        Dense(15, activation = 'relu'),
        Dense(4, activation = 'softmax')    # < softmax activation here
    ]
)
model.compile(
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    optimizer=tf.keras.optimizers.Adam(0.001),
)

model.fit(
    X_train,y_train,
    epochs=10
)
        

Epoch 1/10
63/63 [==============================] - 0s 1ms/step - loss: 1.0319
Epoch 2/10
63/63 [==============================] - 0s 1ms/step - loss: 0.3883
Epoch 3/10
63/63 [==============================] - 0s 1ms/step - loss: 0.1923
Epoch 4/10
63/63 [==============================] - 0s 1ms/step - loss: 0.1181
Epoch 5/10
63/63 [==============================] - 0s 919us/step - loss: 0.0855
Epoch 6/10
63/63 [==============================] - 0s 1ms/step - loss: 0.0701
Epoch 7/10
63/63 [==============================] - 0s 997us/step - loss: 0.0603
Epoch 8/10
63/63 [==============================] - 0s 946us/step - loss: 0.0539
Epoch 9/10
63/63 [==============================] - 0s 1ms/step - loss: 0.0492
Epoch 10/10
63/63 [==============================] - 0s 1ms/step - loss: 0.0454


由于 softmax 已集成到输出层，输出是一个概率向量。

In [8]:
p_nonpreferred = model.predict(X_train)
print(p_nonpreferred [:2])
print("largest value", np.max(p_nonpreferred), "smallest value", np.min(p_nonpreferred))

[[3.36e-03 3.99e-03 9.64e-01 2.89e-02]
 [9.89e-01 1.06e-02 3.22e-04 3.60e-04]]
largest value 0.9999938 smallest value 3.1262097e-08


### 推荐方式 <img align="Right" src="./images/C2_W2_softmax_accurate.png"  style=" width:400px; padding: 10px 20px ; ">
回忆课程内容，如果在训练时把 softmax 和损失合并，能得到更稳定、更准确的结果。这就是这里展示的 '推荐' 组织方式所实现的。

在推荐的组织方式中，最后一层用线性（linear）激活。由于历史原因，这种形式的输出被称为 *logits*。损失函数多了一个参数 `from_logits = True`，它告诉损失函数应把 softmax 运算纳入损失计算，从而实现一个优化过的实现。

In [9]:
preferred_model = Sequential(
    [ 
        Dense(25, activation = 'relu'),
        Dense(15, activation = 'relu'),
        Dense(4, activation = 'linear')   #<-- Note
    ]
)
preferred_model.compile(
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),  #<-- Note
    optimizer=tf.keras.optimizers.Adam(0.001),
)

preferred_model.fit(
    X_train,y_train,
    epochs=10
)
        

Epoch 1/10
63/63 [==============================] - 0s 1ms/step - loss: 1.0041
Epoch 2/10
63/63 [==============================] - 0s 1ms/step - loss: 0.4699
Epoch 3/10
63/63 [==============================] - 0s 1ms/step - loss: 0.2534
Epoch 4/10
63/63 [==============================] - 0s 987us/step - loss: 0.1452
Epoch 5/10
63/63 [==============================] - 0s 1ms/step - loss: 0.0951
Epoch 6/10
63/63 [==============================] - 0s 1ms/step - loss: 0.0711
Epoch 7/10
63/63 [==============================] - 0s 966us/step - loss: 0.0594
Epoch 8/10
63/63 [==============================] - 0s 1ms/step - loss: 0.0531
Epoch 9/10
63/63 [==============================] - 0s 1ms/step - loss: 0.0479
Epoch 10/10
63/63 [==============================] - 0s 1ms/step - loss: 0.0445


#### 输出处理
注意在推荐模型里，输出不是概率，而可能从很大的负数到很大的正数。当做需要概率的预测时，输出必须先经过一个 softmax。
我们看看推荐模型的输出：

In [11]:
p_preferred = preferred_model.predict(X_train)
print(f"two example output vectors:\n {p_preferred[:2]}")
print("largest value", np.max(p_preferred), "smallest value", np.min(p_preferred))

two example output vectors:
 [[-2.95 -3.84  2.8  -1.02]
 [ 4.98  0.07 -4.62 -7.71]]
largest value 10.190824 smallest value -15.386135


这些输出预测不是概率！
如果想要的输出是概率，应把输出经 [softmax](https://www.tensorflow.org/api_docs/python/tf/nn/softmax) 处理。

In [12]:
sm_preferred = tf.nn.softmax(p_preferred).numpy()
print(f"two example output vectors:\n {sm_preferred[:2]}")
print("largest value", np.max(sm_preferred), "smallest value", np.min(sm_preferred))

two example output vectors:
 [[3.10e-03 1.27e-03 9.74e-01 2.13e-02]
 [9.93e-01 7.38e-03 6.75e-05 3.08e-06]]
largest value 0.9999964 smallest value 1.679146e-10


要选出最可能的类别，并不需要 softmax。可以用 [np.argmax()](https://numpy.org/doc/stable/reference/generated/numpy.argmax.html) 找到最大输出的下标。

In [13]:
for i in range(5):
    print( f"{p_preferred[i]}, category: {np.argmax(p_preferred[i])}")

[-2.95 -3.84  2.8  -1.02], category: 2
[ 4.98  0.07 -4.62 -7.71], category: 0
[ 3.58  0.59 -3.7  -6.07], category: 0
[-0.17  4.08 -2.56 -2.13], category: 1
[-1.79 -7.55  3.17 -5.3 ], category: 2


## SparseCategorialCrossentropy 还是 CategoricalCrossEntropy
Tensorflow 对目标值有两种可能的格式，损失函数的选择决定了期望哪种。
- SparseCategorialCrossentropy：期望目标是一个与下标对应的整数。例如若有 10 个可能的目标值，y 在 0 到 9 之间。
- CategoricalCrossEntropy：期望某样本的目标值是 one-hot 编码——目标下标处为 1，其余 N-1 项为零。例如 10 个可能目标值、目标为 2 的样本会是 [0,0,1,0,0,0,0,0,0,0]。

## 恭喜！
在本实验中你：
- 更熟悉了 softmax 函数，及其在 softmax 回归和神经网络 softmax 激活中的使用。
- 学会了 Tensorflow 中推荐的模型构建方式：
    - 最后一层不加激活（等同于线性激活）
    - SparseCategoricalCrossentropy 损失函数
    - 使用 from_logits=True
- 认识到 softmax 与 ReLU、Sigmoid 不同，它横跨多个输出。